# Mineral composition analysis system (XRD)
## Konju National University: Ju Young Park(leahlove6786@gmail.com), Hongkeun Jin. 

### 1. Importing Modules

In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.cluster import KMeans 
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.models import load_model              
import joblib
import warnings

### 2. Data loading

In [2]:
# XRD intensity profile (.csv)
data=pd.read_csv('XRD_weight_change.csv')
X_all=data.iloc[:,13:] 
Y_all=data[["Quartz","Albite", "Opal-A", "Calcite", "Muscovite", "Dolomite", "Chlorite", "Kaolinite", "Illite", "Pyrite", "NaCl","K-feldspar"]];
X_all.shape, Y_all.shape

((488, 3100), (488, 12))

### 3. Data preprocessing
#### 3-1. Sample-based preprocessing (Min-Max scaler)

In [3]:
X_all_T=X_all.T

# MinMaxscaler
df_scale = pd.DataFrame(MinMaxScaler().fit_transform(X_all_T), columns=X_all_T.columns, index = X_all_T.index)

df_scale_T=df_scale.T
df_scale_T.shape

(488, 3100)

#### 3-2. Data split

In [4]:
# split testset
seed = 0
X_tr_val, X_te, Y_tr_val, Y_te = train_test_split(X_all, Y_all, test_size=0.1, random_state=seed)
X_tr_val.shape, X_te.shape, Y_tr_val.shape, Y_te.shape

((439, 3100), (49, 3100), (439, 12), (49, 12))

### 4. Clustering

In [5]:
k = 5
model = KMeans(n_clusters = k, init='k-means++', n_init=100, max_iter=500, tol=1e-04,random_state = 0)

model.fit(X_tr_val)

KMeans(max_iter=500, n_clusters=5, n_init=100, random_state=0)

In [6]:
# Create a DataFrame to store sample numbers and clustering results
clustering_data = pd.DataFrame({'Sample number': range(X_te.shape[0])})

clustering_data['cluster']=model.predict(X_te)

# Changing cluster labels from 0-4 to 1-5
clustering_data['cluster']=clustering_data['cluster']+1

# Expert analysis(0): Cluster 1, 2, 5
# Machine learning analysis(1): Cluster 3, 4
clustering_data['analysis']=clustering_data['cluster'].apply(lambda x: 0 if x in [1, 2, 5] else (1 if x in [3, 4] else None))

# Export to clustering results
df = pd.concat([clustering_data.reset_index(drop=True), X_te.reset_index(drop=True)], axis=1)
df.to_csv('Clustering results_test.csv')
df

,Sample number,cluster,analysis,Angle3.01,Angle3.03,Angle,Angle.1,Angle.2,Angle.3,Angle.4,...,Angle.3088,Angle.3089,Angle.3090,Angle.3091,Angle.3092,Angle.3093,Angle.3094,Angle.3095,Angle64.97,Angle64.99
0,0,4,1,1.520000,41.370000,20.220000,16.070000,-12.080000,-13.230000,3.620000,...,5.036842,-10.968421,2.026316,2.021053,1.015789,6.010526,13.005263,-1.000000,2.000000,1.00
1,1,1,0,2.886080,-3.703693,-1.793466,1.116761,5.026989,-0.062784,-0.652557,...,-0.491445,1.007375,-0.493805,0.505015,-1.996165,0.502655,-0.998525,-0.499705,-1.000000,3.50
2,2,4,1,1.000000,-7.222222,29.555556,4.333333,3.111111,-16.111111,-11.333333,...,7.072464,8.057971,3.043478,-10.971015,2.014493,-2.000000,-2.000000,0.000000,-7.000000,3.00
3,3,4,1,-8.181818,-14.090909,14.000000,-9.909091,5.181818,10.272727,1.363636,...,1.039726,-5.964688,0.030898,5.026484,1.022070,0.017656,2.013242,-0.991172,0.004414,0.00
4,4,5,0,-5.120000,3.410000,15.940000,-17.530000,-25.000000,-22.470000,10.060000,...,-9.980226,-4.983051,4.014124,4.011299,5.008475,-7.994350,-4.997175,4.000000,11.000000,4.00
5,5,1,0,6.511923,1.490385,3.468846,-0.052692,14.925769,-5.095769,-7.117308,...,-1.500000,1.500000,3.000000,1.000000,-1.000000,0.000000,1.500000,1.500000,4.500000,1.50
6,6,4,1,-10.000000,2.652308,15.304615,19.956923,-16.390769,6.261539,-3.086154,...,-0.920755,-4.929560,-3.938365,-0.947170,-1.955975,6.035220,1.026415,-1.982390,2.008805,5.00
7,7,4,1,6.000000,19.816667,9.633333,-8.550000,-17.733333,6.083333,13.900000,...,6.973451,2.976401,4.979351,0.982301,4.985251,7.988201,9.991150,-0.005900,-2.002950,5.00
8,8,2,0,8.043478,1.282609,-6.478261,5.760870,-33.000000,23.239130,4.478261,...,3.059524,6.047619,4.035714,1.023810,-1.988095,0.000000,1.000000,4.000000,-1.000000,1.00
9,9,4,1,-8.900000,5.550000,8.000000,13.450000,12.900000,-2.650000,11.800000,...,0.056497,3.045198,4.033898,2.022599,0.011299,0.000000,-5.000000,3.000000,-5.000000,-3.00


### 5. Mineral composition analysis

### 5-1. Expert group

In [7]:
df_analysis_0 = df[df['analysis'] == 0]
df_analysis_0.to_csv('Analysis by expert.csv', index=False)
df_analysis_0

,Sample number,cluster,analysis,Angle3.01,Angle3.03,Angle,Angle.1,Angle.2,Angle.3,Angle.4,...,Angle.3088,Angle.3089,Angle.3090,Angle.3091,Angle.3092,Angle.3093,Angle.3094,Angle.3095,Angle64.97,Angle64.99
1,1,1,0,2.886080,-3.703693,-1.793466,1.116761,5.026989,-0.062784,-0.652557,...,-0.491445,1.007375,-0.493805,0.505015,-1.996165,0.502655,-0.998525,-0.499705,-1.000000,3.50
4,4,5,0,-5.120000,3.410000,15.940000,-17.530000,-25.000000,-22.470000,10.060000,...,-9.980226,-4.983051,4.014124,4.011299,5.008475,-7.994350,-4.997175,4.000000,11.000000,4.00
5,5,1,0,6.511923,1.490385,3.468846,-0.052692,14.925769,-5.095769,-7.117308,...,-1.500000,1.500000,3.000000,1.000000,-1.000000,0.000000,1.500000,1.500000,4.500000,1.50
8,8,2,0,8.043478,1.282609,-6.478261,5.760870,-33.000000,23.239130,4.478261,...,3.059524,6.047619,4.035714,1.023810,-1.988095,0.000000,1.000000,4.000000,-1.000000,1.00
12,12,1,0,-11.809091,5.954545,8.218182,-0.518182,1.245455,2.009091,4.772727,...,-0.490210,-0.991560,0.507089,0.505739,1.004389,0.503038,0.001688,3.000338,0.500000,2.50
13,13,2,0,-5.978182,4.510909,-8.000000,-11.510909,-17.021818,3.467273,-9.043636,...,-1.927298,-2.935376,-0.943454,-4.951532,-0.959610,5.032312,5.024234,-3.983844,-2.991922,3.00
16,16,2,0,16.019500,11.058000,26.096500,35.135000,-2.826500,-3.788000,34.250500,...,2.600000,-1.400000,-4.400000,7.600000,3.600000,0.600000,2.600000,-1.400000,0.600000,-3.40
18,18,1,0,-0.902143,5.189286,-1.219286,8.872143,-9.536429,3.555000,-0.853571,...,3.500000,0.500000,4.500000,2.000000,2.500000,4.000000,-2.000000,2.500000,-0.500000,3.00
19,19,1,0,0.489091,-0.012727,5.485455,-8.016364,-2.018182,4.480000,2.978182,...,0.527885,2.024038,0.520192,1.016346,1.512500,2.508654,1.004808,-0.999038,1.000000,4.00
20,20,1,0,-9.697368,0.986842,-0.828947,3.355263,5.039474,3.723684,-3.592105,...,0.526079,2.022482,2.518885,-0.984712,1.511691,-0.491907,2.504496,2.500899,-1.500000,1.50


#### 5-2. Machine learning group

In [8]:
df_analysis_1= pd.concat([clustering_data.reset_index(drop=True), X_te.reset_index(drop=True)], axis=1)
df_analysis_1 = df_analysis_1[df_analysis_1['analysis'] == 1]

# export to intensity profile
df_analysis_x=df_analysis_1.iloc[:,3:3104]
df_analysis_x

,Angle3.01,Angle3.03,Angle,Angle.1,Angle.2,Angle.3,Angle.4,Angle.5,Angle.6,Angle.7,...,Angle.3088,Angle.3089,Angle.3090,Angle.3091,Angle.3092,Angle.3093,Angle.3094,Angle.3095,Angle64.97,Angle64.99
0,1.520000,41.370000,20.220000,16.070000,-12.080000,-13.230000,3.620000,-6.530000,4.320000,-2.830000,...,5.036842,-10.968421,2.026316,2.021053,1.015789,6.010526,13.005263,-1.000000,2.000000,1.0
2,1.000000,-7.222222,29.555556,4.333333,3.111111,-16.111111,-11.333333,-9.555556,-7.777778,-14.000000,...,7.072464,8.057971,3.043478,-10.971015,2.014493,-2.000000,-2.000000,0.000000,-7.000000,3.0
3,-8.181818,-14.090909,14.000000,-9.909091,5.181818,10.272727,1.363636,8.454546,-1.454546,5.636364,...,1.039726,-5.964688,0.030898,5.026484,1.022070,0.017656,2.013242,-0.991172,0.004414,0.0
6,-10.000000,2.652308,15.304615,19.956923,-16.390769,6.261539,-3.086154,-16.433846,-14.781538,-8.129231,...,-0.920755,-4.929560,-3.938365,-0.947170,-1.955975,6.035220,1.026415,-1.982390,2.008805,5.0
7,6.000000,19.816667,9.633333,-8.550000,-17.733333,6.083333,13.900000,-10.283333,-2.466667,22.350000,...,6.973451,2.976401,4.979351,0.982301,4.985251,7.988201,9.991150,-0.005900,-2.002950,5.0
9,-8.900000,5.550000,8.000000,13.450000,12.900000,-2.650000,11.800000,-22.750000,31.700000,-9.850000,...,0.056497,3.045198,4.033898,2.022599,0.011299,0.000000,-5.000000,3.000000,-5.000000,-3.0
10,-5.229091,11.385454,-13.000000,5.614546,-12.770909,-5.156364,-20.541818,21.072727,19.687273,-7.698182,...,-4.957640,-3.963691,1.030257,9.024206,2.018154,-2.987897,-2.993949,1.000000,1.000000,-2.0
11,-1.501818,-12.250909,-7.000000,-18.749091,7.501818,-5.247273,7.003636,-16.745455,-0.494546,8.756364,...,14.136364,-0.890909,-1.918182,0.054545,4.027273,5.000000,5.000000,-2.000000,4.000000,4.0
14,4.520000,18.260000,-20.000000,-13.260000,-5.520000,-14.780000,5.960000,-27.300000,1.440000,-15.820000,...,2.019337,3.016575,1.013812,4.011050,-4.991713,0.005525,4.002762,2.000000,10.000000,1.0
15,16.000000,-6.176154,-21.352308,-15.528462,19.295385,3.119231,-26.056923,1.766923,-5.409231,14.414615,...,9.080718,-2.928251,8.062780,2.053812,-4.955157,2.035874,-1.973094,4.017937,3.008969,7.0


In [9]:
composition_model=load_model('./CNN_model_minmax/')                             

In [10]:
composition_model.summary()

Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_18 (Conv1D)          (None, 3051, 32)          1632      
                                                                 
 max_pooling1d_18 (MaxPoolin  (None, 762, 32)          0         
 g1D)                                                            
                                                                 
 conv1d_19 (Conv1D)          (None, 713, 64)           102464    
                                                                 
 max_pooling1d_19 (MaxPoolin  (None, 178, 64)          0         
 g1D)                                                            
                                                                 
 conv1d_20 (Conv1D)          (None, 129, 128)          409728    
                                                                 
 max_pooling1d_20 (MaxPoolin  (None, 32, 128)        

In [11]:
X_scaled_3D = df_analysis_x.values.reshape(df_analysis_x.values.shape[0], df_analysis_x.values.shape[1], 1)

Y_predicted=composition_model.predict(X_te)
CNN_predict = pd.DataFrame(Y_te)
round(CNN_predict.describe().T,2)

2/2 [==============================] - 0s 31ms/step


,count,mean,std,min,25%,50%,75%,max
Quartz,49.0,19.06,9.65,8.1,13.2,16.3,19.0,54.1
Albite,49.0,9.94,5.39,3.3,7.3,8.6,10.3,33.7
Opal-A,49.0,29.21,12.15,0.0,24.7,29.5,36.5,57.5
Calcite,49.0,4.48,7.20,0.0,0.0,2.4,6.0,45.0
Muscovite,49.0,10.81,2.49,4.4,9.4,11.0,12.2,17.6
Dolomite,49.0,0.03,0.21,0.0,0.0,0.0,0.0,1.5
Chlorite,49.0,3.57,1.07,1.1,2.9,3.6,4.3,6.0
Kaolinite,49.0,1.67,0.63,0.0,1.3,1.6,2.1,3.1
Illite,49.0,10.56,2.80,2.2,9.1,11.0,12.4,15.8
Pyrite,49.0,2.68,1.26,0.0,2.1,2.5,3.1,5.7


In [12]:
CNN_predict.to_csv('Analysis by ML.csv')

## End of codes